# Regression and balanced-subsample analyses

Runs the multivariable, subtype-specific, sensitivity, and repeated balanced-subsample analyses.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ======================================================================
# SETUP & DATA
# ======================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats


# --- Load your splits and combine
mimic_train = pd.read_csv(str(DATA_DIR / 'GEP_train_80_20.csv'))
mimic_test  = pd.read_csv(str(DATA_DIR / 'GEP_test_80_20.csv'))
GEP_df = pd.concat([mimic_train, mimic_test], ignore_index=True)

In [ ]:

# ================================================================
# FULL REGRESSION + APPENDIX OUTPUT PIPELINE
# Prints all outputs needed for revised appendix tables:
#   - S3.1 / S3.10: unadjusted models
#   - S3.2a / S3.11a: fully adjusted models
#   - S3.2b / S3.11b: VIFs
#   - S3.3 / S3.12: interaction models
#   - S3.4 / S3.13: model fit comparison
#   - S3.5 / S3.14: sensitivity analyses
#   - S3.6-S3.9: subtype models
#
# Assumes your dataframe is already loaded as GEP_df
# ================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------
# 1. CONFIG
# ------------------------------------------------
USE_COLLAPSED_RACE_FOR_REGRESSION = True

ORIGINAL_RACE_CATEGORIES = [
    'WHITE',
    'BLACK/AFRICAN AMERICAN',
    'ASIAN',
    'HISPANIC/LATINO',
    'OTHER'
]

REGRESSION_RACE_CATEGORIES = [
    'WHITE',
    'BLACK/AFRICAN AMERICAN',
    'HISPANIC/LATINO',
    'OTHER'
]

LANGUAGE_CATEGORIES = ['English', 'Non-English']

# ------------------------------------------------
# 2. PREP HELPERS
# ------------------------------------------------
def standardize_language(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['language_grouped'] = df['language_grouped'].astype(str).str.strip()
    df['language_grouped'] = df['language_grouped'].replace({
        'ENGLISH': 'English',
        'NON-ENGLISH': 'Non-English',
        'English': 'English',
        'Non-English': 'Non-English'
    })
    df['language_grouped'] = pd.Categorical(
        df['language_grouped'],
        categories=LANGUAGE_CATEGORIES,
        ordered=False
    )
    return df


def standardize_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # original race for descriptive use
    df['race_grouped'] = pd.Categorical(
        df['race_grouped'].astype(str).str.strip(),
        categories=ORIGINAL_RACE_CATEGORIES,
        ordered=False
    )

    # collapsed race for regression only
    race_reg = df['race_grouped'].astype(str).replace({
        'ASIAN': 'OTHER'
    })

    df['race_grouped_reg'] = pd.Categorical(
        race_reg,
        categories=REGRESSION_RACE_CATEGORIES,
        ordered=False
    )
    return df


def validate_required_columns(df: pd.DataFrame) -> None:
    required_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'language_grouped',
        'race_grouped',
        'race_grouped_reg',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in df.columns:
        required_cols.append('Misgendering')

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")


def coerce_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    binary_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in df.columns:
        binary_cols.append('Misgendering')

    for col in binary_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df['age_group'] = pd.to_numeric(df['age_group'], errors='coerce').astype(int)
    return df


def build_work_df(df: pd.DataFrame, use_collapsed_race: bool = True):
    df = df.copy()
    df = standardize_language(df)
    df = standardize_race(df)
    validate_required_columns(df)
    df = coerce_numeric_columns(df)

    race_var = 'race_grouped_reg' if use_collapsed_race else 'race_grouped'

    language_dummies = pd.get_dummies(
        df[['language_grouped']],
        drop_first=True
    ).astype(int)

    race_dummies = pd.get_dummies(
        df[[race_var]],
        drop_first=True
    ).astype(int)

    base_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in df.columns:
        base_cols.append('Misgendering')

    work_df = pd.concat([df[base_cols], language_dummies, race_dummies], axis=1).copy()

    protected_cols = {
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering',
        'Misgendering'
    }

    drop_cols = [
        c for c in work_df.columns
        if c not in protected_cols and work_df[c].nunique() <= 1
    ]
    if drop_cols:
        print("Dropping constant columns from work_df:", drop_cols)
        work_df = work_df.drop(columns=drop_cols)

    language_cols = [c for c in language_dummies.columns if c in work_df.columns]
    race_cols = [c for c in race_dummies.columns if c in work_df.columns]
    demo_cols = language_cols + race_cols + ['age_group']

    return df, work_df, language_cols, race_cols, demo_cols


def prepare_X(X: pd.DataFrame) -> pd.DataFrame:
    X = sm.add_constant(X, has_constant='add')
    X = X.apply(pd.to_numeric, errors='coerce').astype(float)
    return X


# ------------------------------------------------
# 3. FIT HELPERS
# ------------------------------------------------
def fit_logit_with_fallback(
    X: pd.DataFrame,
    y: pd.Series,
    alpha: float = 0.01,
    l1_wt: float = 0.0,
    maxiter: int = 500,
    desc: str = "Model"
):
    X = X.copy()
    y = pd.to_numeric(y, errors='coerce').astype(int)

    nunique = X.nunique(dropna=False)
    constant_cols = [c for c in nunique[nunique <= 1].index if c != 'const']
    if constant_cols:
        print(f"[{desc}] Dropping constant columns:", constant_cols)
        X = X.drop(columns=constant_cols)

    X = X.loc[:, ~X.T.duplicated()]

    used_regularized = False

    try:
        model = sm.Logit(y, X)
        res = model.fit(disp=False, maxiter=maxiter)
        converged = res.mle_retvals.get('converged', False)

        if converged:
            bad_params = np.any(~np.isfinite(res.params))
            bad_bse = np.any(~np.isfinite(res.bse))
            huge_bse = np.any(res.bse > 10)
            if bad_params or bad_bse or huge_bse:
                print(f"[{desc}] Standard fit unstable; switching to regularized.")
                raise RuntimeError("Unstable standard fit.")

        return res, used_regularized

    except Exception as e:
        print(f"[{desc}] Standard Logit failed or unstable: {type(e).__name__}: {e}")
        used_regularized = True
        model = sm.Logit(y, X)
        res = model.fit_regularized(
            method='l1',
            alpha=alpha,
            L1_wt=l1_wt,
            maxiter=maxiter
        )
        return res, used_regularized


def extract_or_table(res, order=None) -> pd.DataFrame:
    params = res.params

    try:
        conf = res.conf_int()
        conf.columns = ['lower', 'upper']
    except Exception:
        conf = pd.DataFrame({'lower': np.nan, 'upper': np.nan}, index=params.index)

    try:
        pvals = res.pvalues
    except Exception:
        pvals = pd.Series(np.nan, index=params.index)

    out = pd.DataFrame({
        'Odds Ratio (OR)': np.exp(params),
        'CI_lower': np.exp(conf['lower']),
        'CI_upper': np.exp(conf['upper']),
        'p-value': pvals
    })

    out['95% CI'] = out.apply(
        lambda r: f"{r['CI_lower']:.2f} – {r['CI_upper']:.2f}"
        if pd.notna(r['CI_lower']) and pd.notna(r['CI_upper']) else "NA",
        axis=1
    )

    out['Odds Ratio (OR)'] = out['Odds Ratio (OR)'].round(2)

    if order is not None:
        order = [x for x in order if x in out.index]
        out = out.loc[order]

    return out[['Odds Ratio (OR)', '95% CI', 'p-value']]


def print_named_table(title: str, df: pd.DataFrame):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    print(df.to_string())


def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'Variable': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })


# ------------------------------------------------
# 4. MAIN ANALYSIS FOR ONE OUTCOME VERSION
# ------------------------------------------------
def run_full_appendix_pipeline(
    GEP_df: pd.DataFrame,
    outcome_col: str,
    descriptor_col: str,
    label_name: str,
    sensitivity_title: str,
    use_collapsed_race: bool = True
):
    prepared_df, work_df, language_cols, race_cols, demo_cols = build_work_df(
        GEP_df,
        use_collapsed_race=use_collapsed_race
    )

    y = work_df[outcome_col].astype(int)

    # ----------------------------
    # Unadjusted
    # ----------------------------
    X_unadj = prepare_X(work_df[['GEP']])
    res_unadj, reg_unadj = fit_logit_with_fallback(X_unadj, y, desc=f"{label_name} Unadjusted")

    unadj_order = ['const', 'GEP']
    tab_unadj = extract_or_table(res_unadj, order=unadj_order)
    print_named_table(f"{label_name} - Unadjusted model", tab_unadj)
    print(f"Model summary note: Converged = {not reg_unadj or True}. Regularized fallback = {reg_unadj}")

    # ----------------------------
    # Fully adjusted
    # ----------------------------
    X_adj = prepare_X(work_df[['GEP'] + demo_cols])
    res_adj, reg_adj = fit_logit_with_fallback(X_adj, y, desc=f"{label_name} Fully Adjusted")

    adj_order = ['const', 'GEP'] + language_cols + race_cols + ['age_group']
    tab_adj = extract_or_table(res_adj, order=adj_order)
    print_named_table(f"{label_name} - Fully adjusted model", tab_adj)

    aic_adj = getattr(res_adj, 'aic', np.nan)
    prsquared_adj = getattr(res_adj, 'prsquared', np.nan)
    llf_adj = getattr(res_adj, 'llf', np.nan)
    print(
        f"Model summary note: AIC = {aic_adj:.2f}, "
        f"Pseudo-R² = {prsquared_adj:.4f}, "
        f"Log-Likelihood = {llf_adj:.2f}, "
        f"Converged = True, "
        f"Regularized fallback = {reg_adj}"
    )

    # ----------------------------
    # VIF
    # ----------------------------
    vif_df = compute_vif(X_adj)
    print_named_table(f"{label_name} - Variance Inflation Factors (VIFs)", vif_df.round(3))

    # ----------------------------
    # Interaction models
    # ----------------------------
    interaction_rows = []
    fit_rows = []

    # Language interaction
    res_lang = None
    if len(language_cols) > 0:
        lang_col = language_cols[0]
        X_lang = work_df[['GEP', lang_col] + race_cols + ['age_group']].copy()
        X_lang['GEP_lang_int'] = work_df['GEP'] * work_df[lang_col]
        X_lang = prepare_X(X_lang)
        res_lang, reg_lang = fit_logit_with_fallback(X_lang, y, desc=f"{label_name} GEP x Language")

        if 'GEP_lang_int' in res_lang.params.index:
            interaction_rows.append({
                'Model': 'Model 1: GEP × Language',
                'Interaction': 'GEP × Non-English',
                'Odds Ratio': round(np.exp(res_lang.params['GEP_lang_int']), 2),
                '95% CI': (
                    f"{np.exp(res_lang.conf_int().loc['GEP_lang_int', 0]):.2f} – "
                    f"{np.exp(res_lang.conf_int().loc['GEP_lang_int', 1]):.2f}"
                    if hasattr(res_lang, 'conf_int') else "NA"
                ),
                'p-value': res_lang.pvalues.get('GEP_lang_int', np.nan),
                'Interpretation': (
                    'Smaller GEP effect among non-English patients'
                    if res_lang.pvalues.get('GEP_lang_int', 1) < 0.05
                    else 'No moderation by language'
                )
            })

        fit_rows.append({
            'Model': 'Language Interaction',
            'AIC': getattr(res_lang, 'aic', np.nan),
            'Pseudo-R²': getattr(res_lang, 'prsquared', np.nan),
            'Log-Likelihood': getattr(res_lang, 'llf', np.nan)
        })

    # Age interaction
    X_age = work_df[['GEP', 'age_group'] + language_cols + race_cols].copy()
    X_age['GEP_age_int'] = work_df['GEP'] * work_df['age_group']
    X_age = prepare_X(X_age)
    res_age, reg_age = fit_logit_with_fallback(X_age, y, desc=f"{label_name} GEP x Age")

    interaction_rows.append({
        'Model': 'Model 2: GEP × Age',
        'Interaction': 'GEP × Age group',
        'Odds Ratio': round(np.exp(res_age.params['GEP_age_int']), 2),
        '95% CI': (
            f"{np.exp(res_age.conf_int().loc['GEP_age_int', 0]):.2f} – "
            f"{np.exp(res_age.conf_int().loc['GEP_age_int', 1]):.2f}"
            if hasattr(res_age, 'conf_int') else "NA"
        ),
        'p-value': res_age.pvalues.get('GEP_age_int', np.nan),
        'Interpretation': 'No moderation by age' if res_age.pvalues.get('GEP_age_int', 1) >= 0.05 else 'Moderation by age'
    })

    fit_rows.append({
        'Model': 'Age Interaction',
        'AIC': getattr(res_age, 'aic', np.nan),
        'Pseudo-R²': getattr(res_age, 'prsquared', np.nan),
        'Log-Likelihood': getattr(res_age, 'llf', np.nan)
    })

    # Race interaction
    X_race = work_df[['GEP', 'age_group'] + language_cols + race_cols].copy()
    interaction_name_map = {}
    for col in race_cols:
        int_col = f"GEP_{col}"
        X_race[int_col] = work_df['GEP'] * work_df[col]
        interaction_name_map[int_col] = col
    X_race = prepare_X(X_race)
    res_race, reg_race = fit_logit_with_fallback(X_race, y, desc=f"{label_name} GEP x Race")

    # keep rows for every race interaction term
    for int_col, orig_col in interaction_name_map.items():
        nice_name = orig_col.replace('race_grouped_reg_', '').replace('race_grouped_', '').replace('_', ' ')
        interpretation = 'No moderation by race'
        if res_race.pvalues.get(int_col, 1) < 0.05:
            interpretation = f"Reduced GEP effect in {nice_name.title()} patients"

        interaction_rows.append({
            'Model': 'Model 3: GEP × Race',
            'Interaction': f"GEP × {nice_name.title()}",
            'Odds Ratio': round(np.exp(res_race.params[int_col]), 2),
            '95% CI': (
                f"{np.exp(res_race.conf_int().loc[int_col, 0]):.2f} – "
                f"{np.exp(res_race.conf_int().loc[int_col, 1]):.2f}"
                if hasattr(res_race, 'conf_int') else "NA"
            ),
            'p-value': res_race.pvalues.get(int_col, np.nan),
            'Interpretation': interpretation
        })

    fit_rows.append({
        'Model': 'Race Interaction',
        'AIC': getattr(res_race, 'aic', np.nan),
        'Pseudo-R²': getattr(res_race, 'prsquared', np.nan),
        'Log-Likelihood': getattr(res_race, 'llf', np.nan)
    })

    interaction_df = pd.DataFrame(interaction_rows)
    print_named_table(f"{label_name} - Interaction Models", interaction_df)

    # ----------------------------
    # Model fit comparison
    # ----------------------------
    fit_rows.insert(0, {
        'Model': 'Baseline',
        'AIC': aic_adj,
        'Pseudo-R²': prsquared_adj,
        'Log-Likelihood': llf_adj
    })
    fit_df = pd.DataFrame(fit_rows)
    baseline_aic = fit_df.loc[fit_df['Model'] == 'Baseline', 'AIC'].iloc[0]
    fit_df['ΔAIC vs Baseline'] = fit_df['AIC'] - baseline_aic
    print_named_table(f"{label_name} - Model Fit Comparison", fit_df.round(3))

    # ----------------------------
    # Sensitivity analysis
    # ----------------------------
    sensitivity_mask = ~prepared_df['race_grouped'].isin(['OTHER', 'UNKNOWN'])
    sensitivity_df = prepared_df.loc[sensitivity_mask].copy()
    sens_prepared, sens_work, sens_language_cols, sens_race_cols, sens_demo_cols = build_work_df(
        sensitivity_df,
        use_collapsed_race=use_collapsed_race
    )

    y_sens = sens_work[outcome_col].astype(int)
    X_sens = prepare_X(sens_work[['GEP'] + sens_language_cols + sens_race_cols + ['age_group']])
    res_sens, reg_sens = fit_logit_with_fallback(X_sens, y_sens, desc=f"{label_name} Sensitivity")

    sens_order = ['const', 'GEP'] + sens_language_cols + sens_race_cols + ['age_group']
    tab_sens = extract_or_table(res_sens, order=sens_order)
    print_named_table(f"{label_name} - {sensitivity_title}", tab_sens)

    gep_or_baseline = float(np.exp(res_adj.params['GEP']))
    gep_or_sens = float(np.exp(res_sens.params['GEP']))
    pct_diff = 100 * (gep_or_sens - gep_or_baseline) / gep_or_baseline

    print(
        f"Model summary note: Δ vs Baseline GEP OR = {pct_diff:.1f} %, "
        f"showing {'stable' if abs(pct_diff) < 10 else 'changed'} association; "
        f"Converged = True; Regularized fallback = {reg_sens}."
    )

    # ----------------------------
    # Subtype models
    # ----------------------------
    subtype_names = [
        ('Credibility and Obstinacy', 'Subtype Model 1: Credibility and Obstinacy'),
        ('Compliance', 'Subtype Model 2: Compliance'),
        (descriptor_col, f"Subtype Model 3: {descriptor_col}")
    ]

    # only run Misgendering separately if present; usually not needed for comparative appendix
    for subtype_col, subtype_title in subtype_names:
        y_sub = work_df[subtype_col].astype(int)
        X_sub = prepare_X(work_df[['GEP'] + demo_cols])
        res_sub, reg_sub = fit_logit_with_fallback(X_sub, y_sub, desc=f"{label_name} {subtype_col}")

        sub_order = ['GEP'] + language_cols + race_cols + ['age_group']
        tab_sub = extract_or_table(res_sub, order=sub_order)
        print_named_table(f"{label_name} - {subtype_title}", tab_sub)

        print(
            "Note: Low-frequency race categories were combined into an “Other” category "
            "to ensure stable estimation in regression analyses."
        )

    # return objects if needed
    return {
        'prepared_df': prepared_df,
        'work_df': work_df,
        'res_unadj': res_unadj,
        'res_adj': res_adj,
        'vif_df': vif_df,
        'interaction_df': interaction_df,
        'fit_df': fit_df,
        'res_sens': res_sens
    }


# ------------------------------------------------
# 5. RUN BOTH VERSIONS
# ------------------------------------------------
print("\n" + "#" * 120)
print("RUNNING PIPELINE: INCLUDING MISGENDERING")
print("#" * 120)

results_including = run_full_appendix_pipeline(
    GEP_df=GEP_df,
    outcome_col='label',
    descriptor_col='Descriptors',
    label_name='Including misgendering',
    sensitivity_title='Sensitivity Analysis (Excluding original “Other” and “Unknown” Race)',
    use_collapsed_race=USE_COLLAPSED_RACE_FOR_REGRESSION
)

print("\n" + "#" * 120)
print("RUNNING PIPELINE: EXCLUDING MISGENDERING")
print("#" * 120)

results_excluding = run_full_appendix_pipeline(
    GEP_df=GEP_df,
    outcome_col='label_exclude_misgendering',
    descriptor_col='Descriptors_exclude_misgendering',
    label_name='Excluding misgendering',
    sensitivity_title='Sensitivity Analysis excluding misgendering (Excluding original “Other” and “Unknown” Race)',
    use_collapsed_race=USE_COLLAPSED_RACE_FOR_REGRESSION
)

In [ ]:

# ================================================================
# STANDALONE SENSITIVITY REGRESSION
# Excludes: ASIAN, OTHER, UNKNOWN
# Keeps race as separate categories (no collapsing)
# Outputs one adjusted logistic regression table
# ================================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# ----------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------
LANGUAGE_CATEGORIES = ['English', 'Non-English']
RACE_CATEGORIES_SENSITIVITY = [
    'WHITE',
    'BLACK/AFRICAN AMERICAN',
    'HISPANIC/LATINO'
]

# Choose outcome:
# 'label' for including misgendering
# 'label_exclude_misgendering' for excluding misgendering
OUTCOME_COL = 'label'   # change if needed

# ----------------------------------------------------------------
# 2. PREP
# ----------------------------------------------------------------
df = GEP_df.copy()

# Standardize language
df['language_grouped'] = df['language_grouped'].astype(str).str.strip()
df['language_grouped'] = df['language_grouped'].replace({
    'ENGLISH': 'English',
    'NON-ENGLISH': 'Non-English',
    'English': 'English',
    'Non-English': 'Non-English'
})
df['language_grouped'] = pd.Categorical(
    df['language_grouped'],
    categories=LANGUAGE_CATEGORIES,
    ordered=False
)

# Standardize race
df['race_grouped'] = df['race_grouped'].astype(str).str.strip().str.upper()

# Exclude ASIAN, OTHER, UNKNOWN
df = df.loc[~df['race_grouped'].isin(['ASIAN', 'OTHER', 'UNKNOWN'])].copy()

# Keep only target race categories
df = df.loc[df['race_grouped'].isin(RACE_CATEGORIES_SENSITIVITY)].copy()

df['race_grouped'] = pd.Categorical(
    df['race_grouped'],
    categories=RACE_CATEGORIES_SENSITIVITY,
    ordered=False
)

# Required columns
required_cols = [
    OUTCOME_COL,
    'GEP',
    'language_grouped',
    'race_grouped',
    'age_group'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Numeric conversion
df[OUTCOME_COL] = pd.to_numeric(df[OUTCOME_COL], errors='coerce').fillna(0).astype(int)
df['GEP'] = pd.to_numeric(df['GEP'], errors='coerce').fillna(0).astype(int)
df['age_group'] = pd.to_numeric(df['age_group'], errors='coerce').astype(int)

# Dummies
language_dummies = pd.get_dummies(df[['language_grouped']], drop_first=True).astype(int)
race_dummies = pd.get_dummies(df[['race_grouped']], drop_first=True).astype(int)

work_df = pd.concat([
    df[[OUTCOME_COL, 'GEP', 'age_group']],
    language_dummies,
    race_dummies
], axis=1).copy()

language_cols = [c for c in language_dummies.columns if c in work_df.columns]
race_cols = [c for c in race_dummies.columns if c in work_df.columns]
demo_cols = language_cols + race_cols + ['age_group']

print("Remaining race counts after exclusion:")
print(df['race_grouped'].value_counts(dropna=False))
print("\nLanguage dummy columns:", language_cols)
print("Race dummy columns:", race_cols)

# ----------------------------------------------------------------
# 3. FIT HELPER
# ----------------------------------------------------------------
def fit_logit_with_fallback(X, y, alpha=0.01, l1_wt=0.0, maxiter=500):
    Xc = sm.add_constant(X, has_constant='add').copy()
    Xc = Xc.apply(pd.to_numeric, errors='coerce').astype(float)
    y = pd.to_numeric(y, errors='coerce').astype(int)

    # Drop constant columns except const
    nunique = Xc.nunique(dropna=False)
    constant_cols = [c for c in nunique[nunique <= 1].index if c != 'const']
    if constant_cols:
        Xc = Xc.drop(columns=constant_cols)

    # Drop duplicated columns
    Xc = Xc.loc[:, ~Xc.T.duplicated()]

    try:
        model = sm.Logit(y, Xc)
        res = model.fit(disp=False, maxiter=maxiter)
        converged = res.mle_retvals.get('converged', False)

        # reject unstable standard fits
        if converged:
            bad_params = np.any(~np.isfinite(res.params))
            bad_bse = np.any(~np.isfinite(res.bse))
            huge_bse = np.any(res.bse > 10)
            if bad_params or bad_bse or huge_bse:
                raise RuntimeError("Unstable standard fit")

        return res, False

    except Exception as e:
        print(f"Standard Logit failed or unstable: {type(e).__name__}: {e}")
        model = sm.Logit(y, Xc)
        res = model.fit_regularized(
            method='l1',
            alpha=alpha,
            L1_wt=l1_wt,
            maxiter=maxiter
        )
        return res, True


def extract_or_table(res, order=None):
    params = res.params

    try:
        conf = res.conf_int()
        conf.columns = ['lower', 'upper']
    except Exception:
        conf = pd.DataFrame({'lower': np.nan, 'upper': np.nan}, index=params.index)

    try:
        pvals = res.pvalues
    except Exception:
        pvals = pd.Series(np.nan, index=params.index)

    out = pd.DataFrame({
        'Odds Ratio (OR)': np.exp(params),
        'CI_lower': np.exp(conf['lower']),
        'CI_upper': np.exp(conf['upper']),
        'p-value': pvals
    })

    out['95% CI'] = out.apply(
        lambda r: f"{r['CI_lower']:.2f} – {r['CI_upper']:.2f}"
        if pd.notna(r['CI_lower']) and pd.notna(r['CI_upper']) else "NA",
        axis=1
    )
    out['Odds Ratio (OR)'] = out['Odds Ratio (OR)'].round(2)

    if order is not None:
        order = [x for x in order if x in out.index]
        out = out.loc[order]

    return out[['Odds Ratio (OR)', '95% CI', 'p-value']]

# ----------------------------------------------------------------
# 4. FIT MODEL
# ----------------------------------------------------------------
y = work_df[OUTCOME_COL].astype(int)
X = work_df[['GEP'] + demo_cols].copy()

res, used_regularized = fit_logit_with_fallback(X, y)

order = ['const', 'GEP'] + language_cols + race_cols + ['age_group']
result_table = extract_or_table(res, order=order)

print("\n" + "=" * 100)
print("Sensitivity Analysis (Excluding Asian, Other, and Unknown race)")
print("=" * 100)
print(result_table.to_string())

aic_val = getattr(res, 'aic', np.nan)
pseudo_r2 = getattr(res, 'prsquared', np.nan)
llf_val = getattr(res, 'llf', np.nan)

print("\nModel summary note:")
print(
    f"AIC = {aic_val:.2f}, "
    f"Pseudo-R² = {pseudo_r2:.4f}, "
    f"Log-Likelihood = {llf_val:.2f}, "
    f"Converged = True, "
    f"Regularized fallback = {used_regularized}"
)

In [ ]:

# ================================================================
# STANDALONE SENSITIVITY REGRESSION
# Excludes: ASIAN, OTHER, UNKNOWN
# Keeps race as separate categories (no collapsing)
# Outputs one adjusted logistic regression table
# ================================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# ----------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------
LANGUAGE_CATEGORIES = ['English', 'Non-English']
RACE_CATEGORIES_SENSITIVITY = [
    'WHITE',
    'BLACK/AFRICAN AMERICAN',
    'HISPANIC/LATINO'
]

# Choose outcome:
# 'label' for including misgendering
# 'label_exclude_misgendering' for excluding misgendering
OUTCOME_COL = 'label_exclude_misgendering'   # change if needed

# ----------------------------------------------------------------
# 2. PREP
# ----------------------------------------------------------------
df = GEP_df.copy()

# Standardize language
df['language_grouped'] = df['language_grouped'].astype(str).str.strip()
df['language_grouped'] = df['language_grouped'].replace({
    'ENGLISH': 'English',
    'NON-ENGLISH': 'Non-English',
    'English': 'English',
    'Non-English': 'Non-English'
})
df['language_grouped'] = pd.Categorical(
    df['language_grouped'],
    categories=LANGUAGE_CATEGORIES,
    ordered=False
)

# Standardize race
df['race_grouped'] = df['race_grouped'].astype(str).str.strip().str.upper()

# Exclude ASIAN, OTHER, UNKNOWN
df = df.loc[~df['race_grouped'].isin(['ASIAN', 'OTHER', 'UNKNOWN'])].copy()

# Keep only target race categories
df = df.loc[df['race_grouped'].isin(RACE_CATEGORIES_SENSITIVITY)].copy()

df['race_grouped'] = pd.Categorical(
    df['race_grouped'],
    categories=RACE_CATEGORIES_SENSITIVITY,
    ordered=False
)

# Required columns
required_cols = [
    OUTCOME_COL,
    'GEP',
    'language_grouped',
    'race_grouped',
    'age_group'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Numeric conversion
df[OUTCOME_COL] = pd.to_numeric(df[OUTCOME_COL], errors='coerce').fillna(0).astype(int)
df['GEP'] = pd.to_numeric(df['GEP'], errors='coerce').fillna(0).astype(int)
df['age_group'] = pd.to_numeric(df['age_group'], errors='coerce').astype(int)

# Dummies
language_dummies = pd.get_dummies(df[['language_grouped']], drop_first=True).astype(int)
race_dummies = pd.get_dummies(df[['race_grouped']], drop_first=True).astype(int)

work_df = pd.concat([
    df[[OUTCOME_COL, 'GEP', 'age_group']],
    language_dummies,
    race_dummies
], axis=1).copy()

language_cols = [c for c in language_dummies.columns if c in work_df.columns]
race_cols = [c for c in race_dummies.columns if c in work_df.columns]
demo_cols = language_cols + race_cols + ['age_group']

print("Remaining race counts after exclusion:")
print(df['race_grouped'].value_counts(dropna=False))
print("\nLanguage dummy columns:", language_cols)
print("Race dummy columns:", race_cols)

# ----------------------------------------------------------------
# 3. FIT HELPER
# ----------------------------------------------------------------
def fit_logit_with_fallback(X, y, alpha=0.01, l1_wt=0.0, maxiter=500):
    Xc = sm.add_constant(X, has_constant='add').copy()
    Xc = Xc.apply(pd.to_numeric, errors='coerce').astype(float)
    y = pd.to_numeric(y, errors='coerce').astype(int)

    # Drop constant columns except const
    nunique = Xc.nunique(dropna=False)
    constant_cols = [c for c in nunique[nunique <= 1].index if c != 'const']
    if constant_cols:
        Xc = Xc.drop(columns=constant_cols)

    # Drop duplicated columns
    Xc = Xc.loc[:, ~Xc.T.duplicated()]

    try:
        model = sm.Logit(y, Xc)
        res = model.fit(disp=False, maxiter=maxiter)
        converged = res.mle_retvals.get('converged', False)

        # reject unstable standard fits
        if converged:
            bad_params = np.any(~np.isfinite(res.params))
            bad_bse = np.any(~np.isfinite(res.bse))
            huge_bse = np.any(res.bse > 10)
            if bad_params or bad_bse or huge_bse:
                raise RuntimeError("Unstable standard fit")

        return res, False

    except Exception as e:
        print(f"Standard Logit failed or unstable: {type(e).__name__}: {e}")
        model = sm.Logit(y, Xc)
        res = model.fit_regularized(
            method='l1',
            alpha=alpha,
            L1_wt=l1_wt,
            maxiter=maxiter
        )
        return res, True


def extract_or_table(res, order=None):
    params = res.params

    try:
        conf = res.conf_int()
        conf.columns = ['lower', 'upper']
    except Exception:
        conf = pd.DataFrame({'lower': np.nan, 'upper': np.nan}, index=params.index)

    try:
        pvals = res.pvalues
    except Exception:
        pvals = pd.Series(np.nan, index=params.index)

    out = pd.DataFrame({
        'Odds Ratio (OR)': np.exp(params),
        'CI_lower': np.exp(conf['lower']),
        'CI_upper': np.exp(conf['upper']),
        'p-value': pvals
    })

    out['95% CI'] = out.apply(
        lambda r: f"{r['CI_lower']:.2f} – {r['CI_upper']:.2f}"
        if pd.notna(r['CI_lower']) and pd.notna(r['CI_upper']) else "NA",
        axis=1
    )
    out['Odds Ratio (OR)'] = out['Odds Ratio (OR)'].round(2)

    if order is not None:
        order = [x for x in order if x in out.index]
        out = out.loc[order]

    return out[['Odds Ratio (OR)', '95% CI', 'p-value']]

# ----------------------------------------------------------------
# 4. FIT MODEL
# ----------------------------------------------------------------
y = work_df[OUTCOME_COL].astype(int)
X = work_df[['GEP'] + demo_cols].copy()

res, used_regularized = fit_logit_with_fallback(X, y)

order = ['const', 'GEP'] + language_cols + race_cols + ['age_group']
result_table = extract_or_table(res, order=order)

print("\n" + "=" * 100)
print("Sensitivity Analysis (Excluding Asian, Other, and Unknown race)")
print("=" * 100)
print(result_table.to_string())

aic_val = getattr(res, 'aic', np.nan)
pseudo_r2 = getattr(res, 'prsquared', np.nan)
llf_val = getattr(res, 'llf', np.nan)

print("\nModel summary note:")
print(
    f"AIC = {aic_val:.2f}, "
    f"Pseudo-R² = {pseudo_r2:.4f}, "
    f"Log-Likelihood = {llf_val:.2f}, "
    f"Converged = True, "
    f"Regularized fallback = {used_regularized}"
)

In [ ]:
# ================================================================
# LOGISTIC REGRESSION PIPELINE
# ================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# ------------------------------------------------
# 1. CLEAN / PREP
# ------------------------------------------------
# Make sure language_grouped is exactly what you want
# If it is already correct, this will still be safe.
GEP_df['language_grouped'] = GEP_df['language_grouped'].astype(str).str.strip()
GEP_df['language_grouped'] = GEP_df['language_grouped'].replace({
    'ENGLISH': 'English',
    'NON-ENGLISH': 'Non-English',
    'English': 'English',
    'Non-English': 'Non-English'
})

GEP_df['language_grouped'] = pd.Categorical(
    GEP_df['language_grouped'],
    categories=['English', 'Non-English'],   # baseline first
    ordered=False
)

# Race baseline / order
GEP_df['race_grouped'] = pd.Categorical(
    GEP_df['race_grouped'],
    categories=['WHITE', 'BLACK/AFRICAN AMERICAN', 'ASIAN', 'HISPANIC/LATINO', 'OTHER'],
    ordered=False
)

# Required columns
required_cols = [
    'label',
    'label_exclude_misgendering',
    'GEP',
    'language_grouped',
    'race_grouped',
    'age_group',
    'Credibility and Obstinacy',
    'Compliance',
    'Descriptors',
    'Descriptors_exclude_misgendering'
]

# Add Misgendering only if present
if 'Misgendering' in GEP_df.columns:
    required_cols.append('Misgendering')

missing = [c for c in required_cols if c not in GEP_df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# Binary numeric columns
bin_cols = [
    'label',
    'label_exclude_misgendering',
    'GEP',
    'Credibility and Obstinacy',
    'Compliance',
    'Descriptors',
    'Descriptors_exclude_misgendering'
]
if 'Misgendering' in GEP_df.columns:
    bin_cols.append('Misgendering')

for c in bin_cols:
    GEP_df[c] = pd.to_numeric(GEP_df[c], errors='coerce').fillna(0).astype(int)

# Age ordinal
GEP_df['age_group'] = pd.to_numeric(GEP_df['age_group'], errors='coerce').astype(int)

# Dummies
language_dummies = pd.get_dummies(
    GEP_df[['language_grouped']],
    drop_first=True
)
race_dummies = pd.get_dummies(
    GEP_df[['race_grouped']],
    drop_first=True
)

# Cast dummies
for c in language_dummies.columns:
    language_dummies[c] = language_dummies[c].astype(int)
for c in race_dummies.columns:
    race_dummies[c] = race_dummies[c].astype(int)

# Working frame
base_cols = [
    'label',
    'label_exclude_misgendering',
    'GEP',
    'age_group',
    'Credibility and Obstinacy',
    'Compliance',
    'Descriptors',
    'Descriptors_exclude_misgendering'
]
if 'Misgendering' in GEP_df.columns:
    base_cols.append('Misgendering')

work_df = pd.concat([
    GEP_df[base_cols],
    language_dummies,
    race_dummies
], axis=1).copy()

# Drop constant columns except outcomes
protected_cols = {
    'label',
    'label_exclude_misgendering',
    'GEP',
    'age_group',
    'Credibility and Obstinacy',
    'Compliance',
    'Descriptors',
    'Descriptors_exclude_misgendering',
    'Misgendering'
}
drop_cols = [c for c in work_df.columns if c not in protected_cols and work_df[c].nunique() <= 1]
if drop_cols:
    print("Dropping constant columns from work_df:", drop_cols)
    work_df = work_df.drop(columns=drop_cols)

# Helper lists after dropping constants
language_cols = [c for c in language_dummies.columns if c in work_df.columns]
race_cols = [c for c in race_dummies.columns if c in work_df.columns]
demo_cols = language_cols + race_cols + ['age_group']

print("Language dummy columns:", language_cols)
print("Race dummy columns:", race_cols)

# ------------------------------------------------
# 2. HELPERS
# ------------------------------------------------
def fit_logit_with_fallback(X, y, use_regularized_if_needed=True, alpha=1e-4, l1_wt=0.01, maxiter=200):
    """
    Fit Logit; if standard MLE fails / singular / non-converged,
    fall back to fit_regularized.
    """
    Xc = sm.add_constant(X, has_constant='add').copy()
    Xc = Xc.apply(pd.to_numeric, errors='coerce').astype(float)
    y = pd.to_numeric(y, errors='coerce').astype(int)

    # Drop constant columns except const
    nunique = Xc.nunique(dropna=False)
    constant_cols = [c for c in nunique[nunique <= 1].index if c != 'const']
    if constant_cols:
        print("Dropping constant columns:", constant_cols)
        Xc = Xc.drop(columns=constant_cols)

    # Drop exact duplicate columns
    Xc = Xc.loc[:, ~Xc.T.duplicated()]

    used_regularized = False
    converged = False
    res = None

    # Standard MLE
    try:
        model = sm.Logit(y, Xc)
        res = model.fit(disp=False, maxiter=maxiter)
        converged = res.mle_retvals.get('converged', False)
    except Exception as e:
        print(f"Standard Logit failed: {type(e).__name__}: {e}")
        res = None
        converged = False

    # Fallback
    if (res is None or not converged) and use_regularized_if_needed:
        used_regularized = True
        model = sm.Logit(y, Xc)
        res = model.fit_regularized(method='l1', alpha=alpha, L1_wt=l1_wt, maxiter=maxiter)
        converged = True

        # Try CI / p-values
        try:
            cov = res.cov_params()
            se = np.sqrt(np.diag(cov))
            params = res.params
            zvals = params / se
            pvals = 2 * (1 - stats.norm.cdf(np.abs(zvals)))
            z = stats.norm.ppf(0.975)
            ci_lower = params - z * se
            ci_upper = params + z * se
            conf = pd.DataFrame({'lower': ci_lower, 'upper': ci_upper}, index=params.index)
            return params, conf, pd.Series(pvals, index=params.index), converged, used_regularized
        except Exception:
            params = res.params
            conf = pd.DataFrame({'lower': np.nan, 'upper': np.nan}, index=params.index)
            pvals = pd.Series(np.nan, index=params.index)
            return params, conf, pvals, converged, used_regularized

    # Standard path
    params = res.params
    try:
        conf = res.conf_int()
        conf.columns = ['lower', 'upper']
    except Exception:
        conf = pd.DataFrame({'lower': np.nan, 'upper': np.nan}, index=params.index)

    try:
        pvals = res.pvalues
    except Exception:
        pvals = pd.Series(np.nan, index=params.index)

    return params, conf, pvals, converged, used_regularized


def summarize_or(params, conf_df, pvals, pick=None):
    out = pd.DataFrame({
        'OR': np.exp(params),
        'CI_lower': np.exp(conf_df['lower']),
        'CI_upper': np.exp(conf_df['upper']),
        'p_value': pvals
    })
    if pick is not None:
        out = out.loc[[r for r in pick if r in out.index]]
    return out


def run_main_models(outcome_col, descriptor_col_for_subtype='Descriptors'):
    """
    Runs:
    - unadjusted model
    - adjusted model
    - subtype models
    """
    print("\n" + "="*80)
    print(f"OUTCOME: {outcome_col}")
    print("="*80)

    y = work_df[outcome_col].astype(int)

    # ----------------------------
    # Step 1a: Unadjusted
    # ----------------------------
    X_1a = work_df[['GEP']]
    p1a, c1a, pv1a, conv1a, reg1a = fit_logit_with_fallback(X_1a, y)
    tab1a = summarize_or(p1a, c1a, pv1a, pick=['const', 'GEP'])

    print(f"\n=== Step 1a (UNADJUSTED): {outcome_col} ~ GEP ===")
    print(tab1a.round(4))
    print(f"Converged: {conv1a} | Regularized fallback: {reg1a}")

    # ----------------------------
    # Step 1b: Adjusted
    # ----------------------------
    X_1b = work_df[['GEP'] + demo_cols]
    p1b, c1b, pv1b, conv1b, reg1b = fit_logit_with_fallback(X_1b, y)
    tab1b = summarize_or(p1b, c1b, pv1b)

    print(f"\n=== Step 1b (FULLY ADJUSTED): {outcome_col} ~ GEP + language + race + age ===")
    print(tab1b.round(4))
    print(f"Converged: {conv1b} | Regularized fallback: {reg1b}")

    # ----------------------------
    # H2 OR change
    # ----------------------------
    OR_unadj = np.exp(p1a['GEP'])
    OR_adj = np.exp(p1b['GEP'])
    pct_change = 100.0 * (OR_adj - OR_unadj) / OR_unadj

    print("\n=== H2: GEP OR robustness (unadjusted vs adjusted) ===")
    print(f"GEP OR (unadjusted): {OR_unadj:.4f}")
    print(f"GEP OR (adjusted)  : {OR_adj:.4f}")
    print(f"% change            : {pct_change:+.2f}%")

    # ----------------------------
    # H5 subtype models
    # ----------------------------
    subtypes = [
        'Credibility and Obstinacy',
        'Compliance',
        descriptor_col_for_subtype
    ]
    if 'Misgendering' in work_df.columns:
        subtypes.append('Misgendering')

    rows = []

    for s in subtypes:
        y_s = work_df[s].astype(int)
        X_s = work_df[['GEP'] + demo_cols]

        ps, cs, pvs, convs, regs = fit_logit_with_fallback(X_s, y_s)
        tab_s = summarize_or(ps, cs, pvs)

        print(f"\n=== H5: Subtype-as-outcome: {s} ~ GEP + language + race + age ===")
        print(tab_s.round(4))
        print(f"Converged: {convs} | Regularized fallback: {regs}")

        rows.append({
            'Subtype': s,
            'OR_GEP': tab_s.loc['GEP', 'OR'] if 'GEP' in tab_s.index else np.nan,
            'CI_GEP_low': tab_s.loc['GEP', 'CI_lower'] if 'GEP' in tab_s.index else np.nan,
            'CI_GEP_high': tab_s.loc['GEP', 'CI_upper'] if 'GEP' in tab_s.index else np.nan,
            'p_GEP': tab_s.loc['GEP', 'p_value'] if 'GEP' in tab_s.index else np.nan,
            'Converged': convs,
            'RegularizedFallback': regs
        })

    summary_h5 = pd.DataFrame(rows)

    print("\n=== H5 Summary ===")
    print(summary_h5.round(4))

    return {
        'tab1a': tab1a,
        'tab1b': tab1b,
        'summary_h5': summary_h5,
        'OR_unadj': OR_unadj,
        'OR_adj': OR_adj,
        'pct_change': pct_change
    }

# ------------------------------------------------
# 3. RUN BOTH VERSIONS
# ------------------------------------------------

# INCLUDING misgendering
results_including = run_main_models(
    outcome_col='label',
    descriptor_col_for_subtype='Descriptors'
)

# EXCLUDING misgendering
results_excluding = run_main_models(
    outcome_col='label_exclude_misgendering',
    descriptor_col_for_subtype='Descriptors_exclude_misgendering'
)

In [ ]:
# ================================================================
# ROBUSTNESS + VISUALIZATION PIPELINE
# Runs BOTH versions:
#   1) Including misgendering
#   2) Excluding misgendering
# ================================================================
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="white", font_scale=1.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ----------------------------------------------------------------
# Utility functions
# ----------------------------------------------------------------
def prepare_X_for_logit(X):
    """Ensure matrix is numeric and add constant safely."""
    X = sm.add_constant(X, has_constant='add')
    X = X.apply(pd.to_numeric, errors='coerce').astype(float)
    return X

def safe_logit_fit(X, y, desc="Model"):
    """Fit Logit safely by removing collinear/constant predictors (except const)."""
    X = X.copy()

    nunique = X.nunique()
    constant_cols = [c for c in nunique[nunique <= 1].index if c != 'const']
    if constant_cols:
        print(f"\n[{desc}] Dropping constant columns (excluding const):", constant_cols)
        X = X.drop(columns=constant_cols)

    X = X.loc[:, ~X.T.duplicated()]

    try:
        model = sm.Logit(y, X).fit(disp=False)
    except np.linalg.LinAlgError:
        print(f"[{desc}] Singular matrix detected. Using regularized fit.")
        model = sm.Logit(y, X).fit_regularized(method='l1', alpha=1e-4, maxiter=200)
    return model

def build_work_df(df):
    """
    Build work_df in the same way as the earlier corrected code.
    Assumes language_grouped is already coded as:
      English / Non-English
    """
    tmp = df.copy()

    # enforce categorical baselines
    tmp['language_grouped'] = tmp['language_grouped'].astype(str).str.strip()
    tmp['language_grouped'] = tmp['language_grouped'].replace({
        'ENGLISH': 'English',
        'NON-ENGLISH': 'Non-English',
        'English': 'English',
        'Non-English': 'Non-English'
    })
    tmp['language_grouped'] = pd.Categorical(
        tmp['language_grouped'],
        categories=['English', 'Non-English'],
        ordered=False
    )

    tmp['race_grouped'] = pd.Categorical(
        tmp['race_grouped'],
        categories=['WHITE', 'BLACK/AFRICAN AMERICAN', 'ASIAN', 'HISPANIC/LATINO', 'OTHER'],
        ordered=False
    )

    # numeric columns
    numeric_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in tmp.columns:
        numeric_cols.append('Misgendering')

    for c in numeric_cols:
        if c in tmp.columns:
            tmp[c] = pd.to_numeric(tmp[c], errors='coerce')

    # dummies
    language_dummies = pd.get_dummies(tmp[['language_grouped']], drop_first=True).astype(int)
    race_dummies = pd.get_dummies(tmp[['race_grouped']], drop_first=True).astype(int)

    base_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in tmp.columns:
        base_cols.append('Misgendering')

    work_df = pd.concat([tmp[base_cols], language_dummies, race_dummies], axis=1).copy()

    # drop constant non-core columns
    protected_cols = {
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering',
        'Misgendering'
    }
    drop_cols = [c for c in work_df.columns if c not in protected_cols and work_df[c].nunique() <= 1]
    if drop_cols:
        print("Dropping constant columns from work_df:", drop_cols)
        work_df = work_df.drop(columns=drop_cols)

    language_cols = [c for c in language_dummies.columns if c in work_df.columns]
    race_cols = [c for c in race_dummies.columns if c in work_df.columns]
    demo_cols = language_cols + race_cols + ['age_group']

    return work_df, language_cols, race_cols, demo_cols

# ----------------------------------------------------------------
# Main function for one outcome version
# ----------------------------------------------------------------
def run_robustness_pipeline(
    GEP_df,
    outcome_col,
    descriptor_col,
    baseline_or_gep=None,
    label_name="Including misgendering"
):
    print("\n" + "="*90)
    print(f"ROBUSTNESS PIPELINE: {label_name}")
    print("="*90)

    work_df, language_cols, race_cols, demo_cols = build_work_df(GEP_df)
    y = pd.to_numeric(work_df[outcome_col], errors='coerce').astype(int)

    # choose the language dummy if present
    lang_dummy = None
    if len(language_cols) > 0:
        lang_dummy = language_cols[0]

    # ================================================================
    # 1) INTERACTION CHECKS
    # ================================================================
    # --- GEP × LANGUAGE
    print("\n" + "="*70)
    print(f"=== Interaction Model 1 ({label_name}): GEP × language + race + age ===")

    if lang_dummy is not None:
        work_df['GEP_lang_int'] = work_df['GEP'] * work_df[lang_dummy]
        X_lang_int = work_df[['GEP', lang_dummy, 'GEP_lang_int', 'age_group'] + race_cols].copy()
    else:
        print("Language dummy not available; skipping language interaction model.")
        X_lang_int = None
        res_lang_int = None

    if X_lang_int is not None:
        X_lang_int = prepare_X_for_logit(X_lang_int)
        res_lang_int = safe_logit_fit(X_lang_int, y, desc=f"Interaction (GEP×Language) - {label_name}")
        print(res_lang_int.summary().tables[1])

        if res_lang_int.pvalues.get('GEP_lang_int', 1) < 0.05:
            print(f"\n→ Significant interaction: GEP × {lang_dummy} "
                  f"(OR={np.exp(res_lang_int.params['GEP_lang_int']):.3f}, "
                  f"p={res_lang_int.pvalues['GEP_lang_int']:.4f})")
        else:
            print("\n→ No significant GEP × language interaction detected (p ≥ 0.05).")

    # --- GEP × AGE
    print("\n" + "="*70)
    print(f"=== Interaction Model 2 ({label_name}): GEP × age + language + race ===")

    work_df['GEP_age_int'] = work_df['GEP'] * work_df['age_group']
    X_age_int = work_df[['GEP', 'age_group', 'GEP_age_int'] + language_cols + race_cols].copy()
    X_age_int = prepare_X_for_logit(X_age_int)
    res_age_int = safe_logit_fit(X_age_int, y, desc=f"Interaction (GEP×Age) - {label_name}")

    print(res_age_int.summary().tables[1])
    if res_age_int.pvalues.get('GEP_age_int', 1) < 0.05:
        print(f"\n→ Significant interaction: GEP × age_group "
              f"(OR={np.exp(res_age_int.params['GEP_age_int']):.3f}, "
              f"p={res_age_int.pvalues['GEP_age_int']:.4f})")
    else:
        print("\n→ No significant GEP × age interaction detected (p ≥ 0.05).")

    # --- GEP × RACE
    print("\n" + "="*70)
    print(f"=== Interaction Model 3 ({label_name}): GEP × race + language + age ===")

    race_int_cols = []
    for col in race_cols:
        int_col = f'GEP_{col}'
        work_df[int_col] = work_df['GEP'] * work_df[col]
        race_int_cols.append(int_col)

    X_race_int = work_df[['GEP', 'age_group'] + language_cols + race_cols + race_int_cols].copy()
    X_race_int = prepare_X_for_logit(X_race_int)
    res_race_int = safe_logit_fit(X_race_int, y, desc=f"Interaction (GEP×Race) - {label_name}")

    print(res_race_int.summary().tables[1])
    sig_interactions = [c for c in race_int_cols if res_race_int.pvalues.get(c, 1) < 0.05]
    if sig_interactions:
        print("\n→ Significant GEP × Race interactions detected:")
        for c in sig_interactions:
            print(f"   - {c}: OR={np.exp(res_race_int.params[c]):.3f}, "
                  f"p={res_race_int.pvalues[c]:.4f}")
    else:
        print("\n→ No significant GEP × Race interactions (p ≥ 0.05).")

    # ================================================================
    # 2) SENSITIVITY ANALYSIS
    # ================================================================
    print("\n" + "="*70)
    print(f"=== Sensitivity Model ({label_name}): Excluding 'OTHER' and 'UNKNOWN' Race ===")

    mask = ~GEP_df['race_grouped'].isin(['OTHER', 'UNKNOWN'])
    sensitivity_df = GEP_df.loc[mask].copy()

    sens_work, sens_language_cols, sens_race_cols, _ = build_work_df(sensitivity_df)

    y_sens = pd.to_numeric(sens_work[outcome_col], errors='coerce').astype(int)
    X_sens = sens_work[['GEP', 'age_group'] + sens_language_cols + sens_race_cols].copy()
    X_sens = prepare_X_for_logit(X_sens)
    res_sens = safe_logit_fit(X_sens, y_sens, desc=f"Sensitivity - {label_name}")

    print(res_sens.summary().tables[1])

    if baseline_or_gep is None:
        baseline_or_gep = np.nan

    OR_sens = np.exp(res_sens.params['GEP'])
    pct_diff = 100 * (OR_sens - baseline_or_gep) / baseline_or_gep if pd.notna(baseline_or_gep) else np.nan

    print(f"\nGEP OR (baseline): {baseline_or_gep:.3f}")
    print(f"GEP OR (sensitivity): {OR_sens:.3f}")
    if pd.notna(pct_diff):
        print(f"% difference: {pct_diff:+.2f}%  → {'Stable' if abs(pct_diff)<10 else 'Changed'}")

    # ================================================================
    # 3) MODEL DIAGNOSTICS
    # ================================================================
    print("\n" + "="*70)
    print(f"=== Model Diagnostics for Fully Adjusted Main Model ({label_name}) ===")

    X_diag = work_df[['GEP'] + demo_cols].copy()
    X_diag = prepare_X_for_logit(X_diag)
    y_diag = pd.to_numeric(work_df[outcome_col], errors='coerce').astype(int)
    res_diag = safe_logit_fit(X_diag, y_diag, desc=f"Diagnostics - {label_name}")

    vif_df = pd.DataFrame({
        'Variable': X_diag.columns,
        'VIF': [variance_inflation_factor(X_diag.values, i)
                for i in range(X_diag.shape[1])]
    })

    print("\n--- Variance Inflation Factors ---")
    print(vif_df.round(3))

    print("\n--- Model Fit Statistics ---")
    print(f"AIC: {res_diag.aic:.2f}")
    print(f"McFadden Pseudo-R²: {res_diag.prsquared:.4f}")
    print(f"Log-Likelihood: {res_diag.llf:.2f}")
    print(f"Converged: {res_diag.mle_retvals.get('converged', True)}")

    # ================================================================
    # 4) COMPARATIVE SUMMARY
    # ================================================================
    print("\n" + "="*70)
    print(f"=== Model Comparison ({label_name}): Baseline vs Interaction Models ===")

    models = {
        "Baseline (Step 1b)": res_diag,
        "Age Interaction": res_age_int,
        "Race Interaction": res_race_int,
    }
    if res_lang_int is not None:
        models["Lang Interaction"] = res_lang_int

    summary = pd.DataFrame({
        name: {
            "AIC": m.aic if hasattr(m, "aic") else np.nan,
            "Pseudo-R²": getattr(m, "prsquared", np.nan),
            "Log-Likelihood": getattr(m, "llf", np.nan),
            "Converged": m.mle_retvals.get("converged", True)
        }
        for name, m in models.items()
    }).T

    baseline_aic = summary.loc["Baseline (Step 1b)", "AIC"]
    summary["ΔAIC vs Baseline"] = summary["AIC"] - baseline_aic

    print(summary.round(3))

    best_model = summary["AIC"].idxmin()
    delta_best = summary.loc[best_model, "ΔAIC vs Baseline"]
    print(f"\nBest-fitting model by AIC: {best_model} (ΔAIC = {delta_best:.2f})")
    if abs(delta_best) < 2:
        print("→ Interaction adds negligible improvement (models equivalent).")
    elif abs(delta_best) < 4:
        print("→ Slight improvement with interaction terms.")
    elif abs(delta_best) < 10:
        print("→ Moderate improvement with interaction terms.")
    else:
        print("→ Strong improvement: interaction effects substantially enhance fit.")

    return {
        "work_df": work_df,
        "language_cols": language_cols,
        "race_cols": race_cols,
        "demo_cols": demo_cols,
        "res_diag": res_diag,
        "res_lang_int": res_lang_int,
        "res_age_int": res_age_int,
        "res_race_int": res_race_int,
        "res_sens": res_sens,
        "vif_df": vif_df,
        "summary": summary
    }

# ================================================================
# RUN BOTH VERSIONS
# Requires:
#   results_including['OR_adj']
#   results_excluding['OR_adj']
# from your earlier run_main_models() block
# ================================================================
robust_inc = run_robustness_pipeline(
    GEP_df=GEP_df,
    outcome_col='label',
    descriptor_col='Descriptors',
    baseline_or_gep=results_including['OR_adj'],
    label_name='Including misgendering'
)

robust_exc = run_robustness_pipeline(
    GEP_df=GEP_df,
    outcome_col='label_exclude_misgendering',
    descriptor_col='Descriptors_exclude_misgendering',
    baseline_or_gep=results_excluding['OR_adj'],
    label_name='Excluding misgendering'
)

In [ ]:
# ================================================================
# BALANCED SENSITIVITY ANALYSIS
# Match GEP and NGEP within
# age_group × race_grouped × language_grouped
# ================================================================
import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. Build balanced subset
# ------------------------------------------------
def create_balanced_dataset(df, strata_cols, random_state=42):
    """
    For each stratum, randomly subsample the larger group
    so that #GEP == #NGEP.
    """
    working_df = df.copy()

    # Drop rows with missing values in matching variables
    working_df = working_df.dropna(subset=strata_cols + ['GEP'])

    balanced_parts = []

    grouped = working_df.groupby(strata_cols, observed=False)

    for strata, subset in grouped:
        if subset.empty:
            continue

        gep = subset[subset['GEP'] == 1]
        ngep = subset[subset['GEP'] == 0]

        if len(gep) == 0 or len(ngep) == 0:
            continue

        n = min(len(gep), len(ngep))

        gep_sample = gep.sample(n=n, random_state=random_state)
        ngep_sample = ngep.sample(n=n, random_state=random_state)

        balanced_parts.append(pd.concat([gep_sample, ngep_sample], axis=0))

    if len(balanced_parts) == 0:
        raise ValueError(
            f"No matched strata found for strata_cols={strata_cols}. "
            "Check the column values and overlap between GEP and NGEP."
        )

    balanced_df = pd.concat(balanced_parts, ignore_index=True)
    return balanced_df


# ------------------------------------------------
# 2. Prepare balanced work_df like the main pipeline
# ------------------------------------------------
def build_balanced_work_df(balanced_df):
    # Keep same baselines as main model
    balanced_df = balanced_df.copy()

    balanced_df['language_grouped'] = balanced_df['language_grouped'].astype(str).str.strip()
    balanced_df['language_grouped'] = balanced_df['language_grouped'].replace({
        'ENGLISH': 'English',
        'NON-ENGLISH': 'Non-English',
        'English': 'English',
        'Non-English': 'Non-English'
    })
    balanced_df['language_grouped'] = pd.Categorical(
        balanced_df['language_grouped'],
        categories=['English', 'Non-English'],
        ordered=False
    )

    balanced_df['race_grouped'] = pd.Categorical(
        balanced_df['race_grouped'],
        categories=['WHITE', 'BLACK/AFRICAN AMERICAN', 'ASIAN', 'HISPANIC/LATINO', 'OTHER'],
        ordered=False
    )

    # Binary numeric columns
    bin_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in balanced_df.columns:
        bin_cols.append('Misgendering')

    for c in bin_cols:
        balanced_df[c] = pd.to_numeric(balanced_df[c], errors='coerce').fillna(0).astype(int)

    balanced_df['age_group'] = pd.to_numeric(balanced_df['age_group'], errors='coerce').astype(int)

    # Dummies
    language_dummies = pd.get_dummies(
        balanced_df[['language_grouped']],
        drop_first=True
    ).astype(int)

    race_dummies = pd.get_dummies(
        balanced_df[['race_grouped']],
        drop_first=True
    ).astype(int)

    base_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering'
    ]
    if 'Misgendering' in balanced_df.columns:
        base_cols.append('Misgendering')

    bw = pd.concat([
        balanced_df[base_cols],
        language_dummies,
        race_dummies
    ], axis=1).copy()

    # Drop constant non-protected columns
    protected_cols = {
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group',
        'Credibility and Obstinacy',
        'Compliance',
        'Descriptors',
        'Descriptors_exclude_misgendering',
        'Misgendering'
    }
    drop_cols = [c for c in bw.columns if c not in protected_cols and bw[c].nunique() <= 1]
    if drop_cols:
        print("Dropping constant columns from balanced work_df:", drop_cols)
        bw = bw.drop(columns=drop_cols)

    language_cols_bal = [c for c in language_dummies.columns if c in bw.columns]
    race_cols_bal = [c for c in race_dummies.columns if c in bw.columns]
    demo_cols_bal = language_cols_bal + race_cols_bal + ['age_group']

    return balanced_df, bw, language_cols_bal, race_cols_bal, demo_cols_bal


# ------------------------------------------------
# 3. Run one balanced adjusted model
# ------------------------------------------------
def run_balanced_adjusted_model(bw, demo_cols_bal, outcome_col):
    y = bw[outcome_col].astype(int)
    X = bw[['GEP'] + demo_cols_bal]

    params, conf, pvals, converged, reg = fit_logit_with_fallback(X, y)
    tab = summarize_or(params, conf, pvals)

    return tab, converged, reg


# ------------------------------------------------
# 4. Single-run balanced analysis
# ------------------------------------------------
strata_cols = ['age_group', 'race_grouped', 'language_grouped']

balanced_df = create_balanced_dataset(
    GEP_df,
    strata_cols=strata_cols,
    random_state=42
)

print("\nBalanced counts by stratum and GEP:")
print(
    balanced_df.groupby(['age_group', 'race_grouped', 'language_grouped', 'GEP'], observed=False)
    .size()
)

print("\nBalanced total N:", len(balanced_df))
print("Balanced GEP / NGEP counts:")
print(balanced_df['GEP'].value_counts())

balanced_df, bw, language_cols_bal, race_cols_bal, demo_cols_bal = build_balanced_work_df(balanced_df)

# INCLUDING misgendering
tab_bal_inc, conv_bal_inc, reg_bal_inc = run_balanced_adjusted_model(
    bw, demo_cols_bal, outcome_col='label'
)

print("\n=== Balanced adjusted model: INCLUDING misgendering ===")
print(tab_bal_inc.round(4))
print(f"Converged: {conv_bal_inc} | Regularized fallback: {reg_bal_inc}")

# EXCLUDING misgendering
tab_bal_exc, conv_bal_exc, reg_bal_exc = run_balanced_adjusted_model(
    bw, demo_cols_bal, outcome_col='label_exclude_misgendering'
)

print("\n=== Balanced adjusted model: EXCLUDING misgendering ===")
print(tab_bal_exc.round(4))
print(f"Converged: {conv_bal_exc} | Regularized fallback: {reg_bal_exc}")


# ------------------------------------------------
# 5. Compare full-sample vs balanced ORs
# Assumes you already ran:
# results_including = run_main_models('label', ...)
# results_excluding = run_main_models('label_exclude_misgendering', ...)
# ------------------------------------------------
comparison_rows = []

# Full sample INCLUDING
comparison_rows.append({
    'Analysis': 'Full sample (including misgendering)',
    'OR_GEP': results_including['tab1b'].loc['GEP', 'OR'],
    'CI_low': results_including['tab1b'].loc['GEP', 'CI_lower'],
    'CI_high': results_including['tab1b'].loc['GEP', 'CI_upper'],
    'p_value': results_including['tab1b'].loc['GEP', 'p_value']
})

# Balanced INCLUDING
comparison_rows.append({
    'Analysis': 'Balanced sample (including misgendering)',
    'OR_GEP': tab_bal_inc.loc['GEP', 'OR'],
    'CI_low': tab_bal_inc.loc['GEP', 'CI_lower'],
    'CI_high': tab_bal_inc.loc['GEP', 'CI_upper'],
    'p_value': tab_bal_inc.loc['GEP', 'p_value']
})

# Full sample EXCLUDING
comparison_rows.append({
    'Analysis': 'Full sample (excluding misgendering)',
    'OR_GEP': results_excluding['tab1b'].loc['GEP', 'OR'],
    'CI_low': results_excluding['tab1b'].loc['GEP', 'CI_lower'],
    'CI_high': results_excluding['tab1b'].loc['GEP', 'CI_upper'],
    'p_value': results_excluding['tab1b'].loc['GEP', 'p_value']
})

# Balanced EXCLUDING
comparison_rows.append({
    'Analysis': 'Balanced sample (excluding misgendering)',
    'OR_GEP': tab_bal_exc.loc['GEP', 'OR'],
    'CI_low': tab_bal_exc.loc['GEP', 'CI_lower'],
    'CI_high': tab_bal_exc.loc['GEP', 'CI_upper'],
    'p_value': tab_bal_exc.loc['GEP', 'p_value']
})

comparison_df = pd.DataFrame(comparison_rows)

print("\n=== Full-sample vs balanced-sample comparison ===")
print(comparison_df.round(4))

In [ ]:
(balanced_df.groupby(['age_group','race_grouped','language_grouped','GEP']).size().unstack(fill_value=0).nunique(axis=1) == 1).all()

In [ ]:
balanced_df.groupby(['age_group','race_grouped','language_grouped','GEP']).size().unstack(fill_value=0)

In [ ]:
# ================================================================
# REPEATED BALANCED SUBSAMPLING SENSITIVITY ANALYSIS
# Matches GEP and NGEP within:
#   age_group × race_grouped × language_grouped
# Repeats many times and summarizes OR(GEP)
# ================================================================
import pandas as pd
import numpy as np

def create_balanced_dataset(df, strata_cols, random_state=42):
    working_df = df.copy()
    working_df = working_df.dropna(subset=strata_cols + ['GEP'])

    balanced_parts = []
    grouped = working_df.groupby(strata_cols, observed=False)

    for _, subset in grouped:
        gep = subset[subset['GEP'] == 1]
        ngep = subset[subset['GEP'] == 0]

        if len(gep) == 0 or len(ngep) == 0:
            continue

        n = min(len(gep), len(ngep))
        gep_sample = gep.sample(n=n, random_state=random_state)
        ngep_sample = ngep.sample(n=n, random_state=random_state)

        balanced_parts.append(pd.concat([gep_sample, ngep_sample], axis=0))

    if len(balanced_parts) == 0:
        raise ValueError("No matched strata found. Check strata columns and overlap.")

    return pd.concat(balanced_parts, ignore_index=True)


def build_work_df_for_balanced(df):
    working_df = df.copy()

    # Ensure categorical baselines match your main analysis
    working_df['language_grouped'] = working_df['language_grouped'].astype(str).str.strip()
    working_df['language_grouped'] = working_df['language_grouped'].replace({
        'ENGLISH': 'English',
        'NON-ENGLISH': 'Non-English',
        'English': 'English',
        'Non-English': 'Non-English'
    })
    working_df['language_grouped'] = pd.Categorical(
        working_df['language_grouped'],
        categories=['English', 'Non-English'],
        ordered=False
    )

    working_df['race_grouped'] = pd.Categorical(
        working_df['race_grouped'],
        categories=['WHITE', 'BLACK/AFRICAN AMERICAN', 'ASIAN', 'HISPANIC/LATINO', 'OTHER'],
        ordered=False
    )

    # Numeric conversion
    numeric_cols = [
        'label',
        'label_exclude_misgendering',
        'GEP',
        'age_group'
    ]
    for c in numeric_cols:
        working_df[c] = pd.to_numeric(working_df[c], errors='coerce')

    # Dummies
    language_dummies = pd.get_dummies(working_df[['language_grouped']], drop_first=True).astype(int)
    race_dummies = pd.get_dummies(working_df[['race_grouped']], drop_first=True).astype(int)

    work = pd.concat([
        working_df[['label', 'label_exclude_misgendering', 'GEP', 'age_group']],
        language_dummies,
        race_dummies
    ], axis=1).copy()

    # Drop constant non-core columns
    protected = {'label', 'label_exclude_misgendering', 'GEP', 'age_group'}
    drop_cols = [c for c in work.columns if c not in protected and work[c].nunique() <= 1]
    if drop_cols:
        work = work.drop(columns=drop_cols)

    language_cols = [c for c in language_dummies.columns if c in work.columns]
    race_cols = [c for c in race_dummies.columns if c in work.columns]
    demo_cols = language_cols + race_cols + ['age_group']

    return work, demo_cols


def run_one_balanced_model(df, outcome_col, strata_cols, seed):
    balanced_df = create_balanced_dataset(df, strata_cols=strata_cols, random_state=seed)
    work_bal, demo_cols_bal = build_work_df_for_balanced(balanced_df)

    y = work_bal[outcome_col].astype(int)
    X = work_bal[['GEP'] + demo_cols_bal]

    params, conf, pvals, converged, reg = fit_logit_with_fallback(X, y)
    tab = summarize_or(params, conf, pvals)

    return {
        'seed': seed,
        'n_total': len(balanced_df),
        'n_gep': int((balanced_df['GEP'] == 1).sum()),
        'n_ngep': int((balanced_df['GEP'] == 0).sum()),
        'OR_GEP': tab.loc['GEP', 'OR'],
        'CI_low': tab.loc['GEP', 'CI_lower'],
        'CI_high': tab.loc['GEP', 'CI_upper'],
        'p_value': tab.loc['GEP', 'p_value'],
        'Converged': converged,
        'RegularizedFallback': reg
    }


def repeat_balanced_analysis(df, outcome_col, strata_cols, n_repeats=100, seed_start=1000):
    rows = []

    for i in range(n_repeats):
        seed = seed_start + i
        try:
            out = run_one_balanced_model(df, outcome_col, strata_cols, seed)
            rows.append(out)
        except Exception as e:
            rows.append({
                'seed': seed,
                'n_total': np.nan,
                'n_gep': np.nan,
                'n_ngep': np.nan,
                'OR_GEP': np.nan,
                'CI_low': np.nan,
                'CI_high': np.nan,
                'p_value': np.nan,
                'Converged': False,
                'RegularizedFallback': np.nan,
                'Error': str(e)
            })

    results = pd.DataFrame(rows)
    return results


def summarize_repeated_results(results_df, label):
    valid = results_df.dropna(subset=['OR_GEP']).copy()

    summary = pd.DataFrame([{
        'Analysis': label,
        'Runs': len(results_df),
        'ValidRuns': len(valid),
        'Mean_OR_GEP': valid['OR_GEP'].mean(),
        'SD_OR_GEP': valid['OR_GEP'].std(),
        'Median_OR_GEP': valid['OR_GEP'].median(),
        'P2.5_OR_GEP': valid['OR_GEP'].quantile(0.025),
        'P97.5_OR_GEP': valid['OR_GEP'].quantile(0.975),
        'Mean_p_value': valid['p_value'].mean(),
        'Prop_p_lt_0.05': (valid['p_value'] < 0.05).mean(),
        'Mean_N': valid['n_total'].mean()
    }])

    return summary


# ------------------------------------------------
# RUN THE REPEATED ANALYSES
# ------------------------------------------------
strata_cols = ['age_group', 'race_grouped', 'language_grouped']

# INCLUDING misgendering
rep_inc = repeat_balanced_analysis(
    GEP_df,
    outcome_col='label',
    strata_cols=strata_cols,
    n_repeats=100,
    seed_start=1000
)

# EXCLUDING misgendering
rep_exc = repeat_balanced_analysis(
    GEP_df,
    outcome_col='label_exclude_misgendering',
    strata_cols=strata_cols,
    n_repeats=100,
    seed_start=2000
)

summary_inc = summarize_repeated_results(rep_inc, 'Balanced repeated analysis (including misgendering)')
summary_exc = summarize_repeated_results(rep_exc, 'Balanced repeated analysis (excluding misgendering)')

summary_both = pd.concat([summary_inc, summary_exc], ignore_index=True)

print("\n=== Repeated balanced sensitivity summary ===")
print(summary_both.round(4))

print("\n=== First few runs: INCLUDING misgendering ===")
print(rep_inc.head().round(4))

print("\n=== First few runs: EXCLUDING misgendering ===")
print(rep_exc.head().round(4))

In [ ]:
rep_inc.to_csv(str(DATA_DIR / 'balanced_runs_including_misgendering.csv'), index=False)
rep_exc.to_csv(str(DATA_DIR / 'balanced_runs_excluding_misgendering.csv'), index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plt.rcParams["font.family"] = "sans-serif"
save_dir = str(FIGURES_DIR) + os.sep

sns.set(style="white", font_scale=1.0)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

plot_df = pd.concat([
    rep_inc[['OR_GEP']].assign(Analysis='Including Misgendering'),
    rep_exc[['OR_GEP']].assign(Analysis='Excluding Misgendering')
], ignore_index=True)

palette = {
    "Including Misgendering": "#2b8cbe",
    "Excluding Misgendering": "#7bccc4"
}

median_inc = rep_inc['OR_GEP'].median()
median_exc = rep_exc['OR_GEP'].median()

fig, ax = plt.subplots(figsize=(8, 6))

sns.kdeplot(
    data=plot_df,
    x="OR_GEP",
    hue="Analysis",
    palette=palette,
    fill=True,
    common_norm=False,
    alpha=0.45,
    linewidth=2,
    ax=ax
)

# Median lines
ax.axvline(median_inc, color="#2b8cbe", linestyle="--", linewidth=2)
ax.axvline(median_exc, color="#7bccc4", linestyle="--", linewidth=2)

# Get y-axis max
ymax = ax.get_ylim()[1]

# Put labels INSIDE the plot, near the top, away from title
ax.text(
    median_exc,
    ymax * 0.98,
    f"Median OR = {median_exc:.2f}",
    color="#7bccc4",
    ha='center',
    va='bottom',
    fontsize=10, # Adjusted from 10
    fontweight='bold',
    bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2)
)

ax.text(
    median_inc,
    ymax * 0.98,
    f"Median OR = {median_inc:.2f}",
    color="#2b8cbe",
    ha='center',
    va='bottom',
    fontsize=10, # Adjusted from 10
    fontweight='bold',
    bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2)
)

ax.set_xlabel("Odds Ratio for GEP Status")
ax.set_ylabel("Density")
ax.set_title(
    "Distribution of GEP Odds Ratios Across 100 Balanced Subsamples",
    pad=20
)

# Access and modify the existing legend
legend = ax.get_legend()
if legend:
    for text in legend.get_texts():
        text.set_fontsize(8) # Adjusted from 10
    if legend.get_title():
        legend.get_title().set_fontsize(8) # Adjusted from 10
        legend.set_title('') # Remove the legend title
    legend.set_bbox_to_anchor((0.5, 1.0))
    legend.set_loc('upper center')

plt.tight_layout()

plt.savefig(
    save_dir + "balanced_OR_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    save_dir + "balanced_OR_distribution.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


data_dir = str(DATA_DIR) + os.sep
save_dir = str(FIGURES_DIR) + os.sep
MAX_PNG_DIMENSION = 1200

rep_inc = pd.read_csv(
    data_dir + "balanced_runs_including_misgendering.csv"
)
rep_exc = pd.read_csv(
    data_dir + "balanced_runs_excluding_misgendering.csv"
)

sns.set(style="white", font_scale=1.0)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

plot_df = pd.concat(
    [
        rep_inc[["OR_GEP"]].assign(Analysis="Including misgendering"),
        rep_exc[["OR_GEP"]].assign(Analysis="Excluding misgendering"),
    ],
    ignore_index=True,
)

palette = {
    "Including misgendering": "#2b8cbe",
    "Excluding misgendering": "#7bccc4",
}

median_inc = rep_inc["OR_GEP"].median()
median_exc = rep_exc["OR_GEP"].median()

fig, ax = plt.subplots(figsize=(8, 6))

sns.kdeplot(
    data=plot_df,
    x="OR_GEP",
    hue="Analysis",
    palette=palette,
    fill=True,
    common_norm=False,
    alpha=0.45,
    linewidth=2,
    ax=ax,
)

# Median lines
ax.axvline(median_inc, color="#2b8cbe", linestyle="--", linewidth=2)
ax.axvline(median_exc, color="#7bccc4", linestyle="--", linewidth=2)

# Get y-axis maximum
ymax = ax.get_ylim()[1]

# Place labels inside the plot near the top
ax.text(
    median_exc,
    ymax * 0.98,
    f"Median OR={median_exc:.2f}",
    color="#7bccc4",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2),
)

ax.text(
    median_inc,
    ymax * 0.98,
    f"Median OR={median_inc:.2f}",
    color="#2b8cbe",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2),
)

ax.set_xlabel("OR for GEP status")
ax.set_ylabel("Density")
ax.set_title(
    "Distribution of GEP ORs across 100 balanced subsamples",
    pad=20,
)

# Access and modify the existing legend
legend = ax.get_legend()
if legend:
    for text in legend.get_texts():
        text.set_fontsize(8)
    legend.set_title("")
    legend.set_bbox_to_anchor((0.5, 1.0))
    if hasattr(legend, "set_loc"):
        legend.set_loc("upper center")
    else:
        legend._loc = 9  # Matplotlib location code for "upper center"

plt.tight_layout()

# Preserve the original 8 x 6 inch layout while limiting the PNG to
# 1200 pixels on its longest side, as required by JMIR.
png_dpi = min(600, MAX_PNG_DIMENSION / max(fig.get_size_inches()))
fig.savefig(
    save_dir + "balanced_OR_distribution.png",
    dpi=png_dpi,
    bbox_inches=None,
    pad_inches=0,
    facecolor="white",
    transparent=False,
)

fig.savefig(
    save_dir + "balanced_OR_distribution.pdf",
    bbox_inches="tight",
)

plt.show()


In [ ]:
# ================================================================
# LOGISTIC REGRESSION HELPERS
# Required by diagnostic balanced-subsampling code
# ================================================================
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationError


def fit_logit_with_fallback(X, y, alpha=1e-6, maxiter=5000):
    """
    Fit standard logistic regression first.
    If standard Logit fails or does not converge, fall back to L1-regularized Logit.

    Returns:
        params: pandas Series
        conf: pandas DataFrame with columns ["CI_lower", "CI_upper"]
        pvals: pandas Series
        converged: bool
        regularized_fallback: bool
    """
    X = X.copy()
    y = pd.to_numeric(y, errors="coerce")

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = sm.add_constant(X, has_constant="add")

    valid_mask = y.notna() & X.notna().all(axis=1)
    y = y.loc[valid_mask].astype(float)
    X = X.loc[valid_mask].astype(float)

    model = sm.Logit(y, X)

    # ------------------------------------------------------------
    # First try standard maximum-likelihood Logit
    # ------------------------------------------------------------
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            result = model.fit(disp=False, maxiter=maxiter)

        converged = bool(result.mle_retvals.get("converged", False))

        if not converged:
            raise RuntimeError("Standard Logit did not converge")

        params = result.params
        conf_raw = result.conf_int()
        conf = pd.DataFrame({
            "CI_lower": conf_raw[0],
            "CI_upper": conf_raw[1]
        })
        pvals = result.pvalues

        return params, conf, pvals, True, False

    except Exception as e:
        print(f"Standard Logit failed: {type(e).__name__}: {e}")

    # ------------------------------------------------------------
    # Fallback: L1-regularized Logit
    # ------------------------------------------------------------
    try:
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            reg_result = model.fit_regularized(
                method="l1",
                alpha=alpha,
                maxiter=maxiter,
                disp=False,
                trim_mode="off"
            )

        params = pd.Series(reg_result.params, index=X.columns)

        # Approximate covariance using inverse Hessian at penalized estimate.
        # This is a pragmatic fallback for quasi-separation/sparse strata.
        try:
            hess = model.hessian(params.values)
            cov = np.linalg.inv(-hess)
            se = pd.Series(np.sqrt(np.diag(cov)), index=X.columns)
        except Exception:
            se = pd.Series(np.nan, index=X.columns)

        z = params / se
        pvals = pd.Series(
            2 * (1 - stats.norm.cdf(np.abs(z))),
            index=X.columns
        )

        conf = pd.DataFrame({
            "CI_lower": params - 1.96 * se,
            "CI_upper": params + 1.96 * se
        })

        return params, conf, pvals, True, True

    except Exception as e:
        raise RuntimeError(f"Regularized fallback also failed: {type(e).__name__}: {e}")


def summarize_or(params, conf, pvals):
    """
    Convert log-odds coefficients to OR table.
    """
    tab = pd.DataFrame({
        "coef": params,
        "OR": np.exp(params),
        "CI_lower": np.exp(conf["CI_lower"]),
        "CI_upper": np.exp(conf["CI_upper"]),
        "p_value": pvals
    })

    return tab

In [ ]:
# ================================================================
# REPEATED BALANCED SUBSAMPLING SENSITIVITY ANALYSIS
# WITH RUN-LEVEL + STRATUM-LEVEL DIAGNOSTICS
# ================================================================
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

OUT_DIR = str(DATA_DIR) + os.sep

# ------------------------------------------------
# Helpers assumed from your existing code:
#   fit_logit_with_fallback(X, y)
#   summarize_or(params, conf, pvals)
#
# If they are already defined above, do not redefine them.
# ------------------------------------------------

def create_balanced_dataset_with_diagnostics(df, strata_cols, random_state=42):
    working_df = df.copy()
    working_df = working_df.dropna(subset=strata_cols + ["GEP"])

    balanced_parts = []
    strata_rows = []

    grouped = working_df.groupby(strata_cols, observed=True, dropna=True)

    for stratum_key, subset in grouped:
        if not isinstance(stratum_key, tuple):
            stratum_key = (stratum_key,)

        gep = subset[subset["GEP"] == 1]
        ngep = subset[subset["GEP"] == 0]

        n_gep_available = len(gep)
        n_ngep_available = len(ngep)
        retained = n_gep_available > 0 and n_ngep_available > 0
        n_pairs = min(n_gep_available, n_ngep_available) if retained else 0

        row = {col: val for col, val in zip(strata_cols, stratum_key)}
        row.update({
            "n_gep_available": n_gep_available,
            "n_ngep_available": n_ngep_available,
            "retained": retained,
            "n_pairs_sampled": n_pairs,
            "n_total_sampled": 2 * n_pairs
        })
        strata_rows.append(row)

        if retained:
            gep_sample = gep.sample(n=n_pairs, random_state=random_state)
            ngep_sample = ngep.sample(n=n_pairs, random_state=random_state)
            balanced_parts.append(pd.concat([gep_sample, ngep_sample], axis=0))

    if len(balanced_parts) == 0:
        raise ValueError("No matched strata found. Check strata columns and overlap.")

    balanced_df = pd.concat(balanced_parts, ignore_index=True)
    strata_diag = pd.DataFrame(strata_rows)

    return balanced_df, strata_diag


def build_work_df_for_balanced_diagnostics(df):
    working_df = df.copy()

    working_df["language_grouped"] = working_df["language_grouped"].astype(str).str.strip()
    working_df["language_grouped"] = working_df["language_grouped"].replace({
        "ENGLISH": "English",
        "NON-ENGLISH": "Non-English",
        "English": "English",
        "Non-English": "Non-English"
    })
    working_df["language_grouped"] = pd.Categorical(
        working_df["language_grouped"],
        categories=["English", "Non-English"],
        ordered=False
    )

    working_df["race_grouped"] = working_df["race_grouped"].astype(str).str.strip()
    working_df["race_grouped"] = working_df["race_grouped"].replace({
        "White": "WHITE",
        "WHITE": "WHITE",
        "Black/African American": "BLACK/AFRICAN AMERICAN",
        "BLACK/AFRICAN AMERICAN": "BLACK/AFRICAN AMERICAN",
        "Asian": "ASIAN",
        "ASIAN": "ASIAN",
        "Hispanic/Latino": "HISPANIC/LATINO",
        "HISPANIC/LATINO": "HISPANIC/LATINO",
        "Other": "OTHER",
        "OTHER": "OTHER"
    })
    working_df["race_grouped"] = pd.Categorical(
        working_df["race_grouped"],
        categories=[
            "WHITE",
            "BLACK/AFRICAN AMERICAN",
            "ASIAN",
            "HISPANIC/LATINO",
            "OTHER"
        ],
        ordered=False
    )

    numeric_cols = [
        "label",
        "label_exclude_misgendering",
        "GEP",
        "age_group"
    ]
    for c in numeric_cols:
        working_df[c] = pd.to_numeric(working_df[c], errors="coerce")

    language_dummies_all = pd.get_dummies(
        working_df[["language_grouped"]],
        drop_first=True
    ).astype(int)

    race_dummies_all = pd.get_dummies(
        working_df[["race_grouped"]],
        drop_first=True
    ).astype(int)

    work = pd.concat([
        working_df[["label", "label_exclude_misgendering", "GEP", "age_group"]],
        language_dummies_all,
        race_dummies_all
    ], axis=1).copy()

    protected = {"label", "label_exclude_misgendering", "GEP", "age_group"}
    candidate_demo_cols = [
        c for c in work.columns
        if c not in protected
    ] + ["age_group"]

    drop_cols = [
        c for c in work.columns
        if c not in protected and work[c].nunique(dropna=True) <= 1
    ]

    if drop_cols:
        work = work.drop(columns=drop_cols)

    language_cols = [c for c in language_dummies_all.columns if c in work.columns]
    race_cols = [c for c in race_dummies_all.columns if c in work.columns]
    demo_cols = language_cols + race_cols + ["age_group"]

    dropped_demo_cols = [c for c in candidate_demo_cols if c not in demo_cols]

    return work, demo_cols, demo_cols, dropped_demo_cols


def summarize_retained_levels(balanced_df, strata_cols):
    out = {}

    for col in strata_cols:
        levels = sorted([str(x) for x in balanced_df[col].dropna().unique()])
        out[f"n_{col}_levels_retained"] = len(levels)
        out[f"{col}_levels_retained"] = "; ".join(levels)

    return out


def summarize_strata(strata_diag):
    retained = strata_diag[strata_diag["retained"]].copy()

    if retained.empty:
        return {
            "n_matched_strata": 0,
            "min_pairs_per_stratum": np.nan,
            "median_pairs_per_stratum": np.nan,
            "max_pairs_per_stratum": np.nan,
            "min_total_per_stratum": np.nan,
            "median_total_per_stratum": np.nan,
            "max_total_per_stratum": np.nan
        }

    return {
        "n_matched_strata": int(len(retained)),
        "min_pairs_per_stratum": int(retained["n_pairs_sampled"].min()),
        "median_pairs_per_stratum": float(retained["n_pairs_sampled"].median()),
        "max_pairs_per_stratum": int(retained["n_pairs_sampled"].max()),
        "min_total_per_stratum": int(retained["n_total_sampled"].min()),
        "median_total_per_stratum": float(retained["n_total_sampled"].median()),
        "max_total_per_stratum": int(retained["n_total_sampled"].max())
    }


def run_one_balanced_model_with_diagnostics(df, outcome_col, strata_cols, seed, analysis_label):
    balanced_df, strata_diag = create_balanced_dataset_with_diagnostics(
        df,
        strata_cols=strata_cols,
        random_state=seed
    )

    work_bal, demo_cols_bal, retained_demo_cols, dropped_demo_cols = (
        build_work_df_for_balanced_diagnostics(balanced_df)
    )

    y = work_bal[outcome_col].astype(int)
    X = work_bal[["GEP"] + demo_cols_bal]

    params, conf, pvals, converged, reg = fit_logit_with_fallback(X, y)
    tab = summarize_or(params, conf, pvals)

    level_summary = summarize_retained_levels(balanced_df, strata_cols)
    strata_summary = summarize_strata(strata_diag)

    run_row = {
        "analysis": analysis_label,
        "seed": seed,
        "n_total": len(balanced_df),
        "n_gep": int((balanced_df["GEP"] == 1).sum()),
        "n_ngep": int((balanced_df["GEP"] == 0).sum()),
        **level_summary,
        **strata_summary,
        "demo_cols_retained": "; ".join(retained_demo_cols),
        "demo_cols_dropped": "; ".join(dropped_demo_cols),
        "n_demo_cols_retained": len(retained_demo_cols),
        "n_demo_cols_dropped": len(dropped_demo_cols),
        "OR_GEP": tab.loc["GEP", "OR"],
        "CI_low": tab.loc["GEP", "CI_lower"],
        "CI_high": tab.loc["GEP", "CI_upper"],
        "p_value": tab.loc["GEP", "p_value"],
        "Converged": converged,
        "RegularizedFallback": reg,
        "Error": ""
    }

    strata_diag = strata_diag.copy()
    strata_diag.insert(0, "analysis", analysis_label)
    strata_diag.insert(1, "seed", seed)

    return run_row, strata_diag


def repeat_balanced_analysis_with_diagnostics(
    df,
    outcome_col,
    strata_cols,
    analysis_label,
    n_repeats=100,
    seed_start=1000
):
    run_rows = []
    strata_tables = []

    for i in range(n_repeats):
        seed = seed_start + i

        try:
            run_row, strata_diag = run_one_balanced_model_with_diagnostics(
                df=df,
                outcome_col=outcome_col,
                strata_cols=strata_cols,
                seed=seed,
                analysis_label=analysis_label
            )

            run_rows.append(run_row)
            strata_tables.append(strata_diag)

        except Exception as e:
            print(f"[{analysis_label}] seed={seed} failed: {type(e).__name__}: {e}")

            run_rows.append({
                "analysis": analysis_label,
                "seed": seed,
                "n_total": np.nan,
                "n_gep": np.nan,
                "n_ngep": np.nan,
                "n_age_group_levels_retained": np.nan,
                "age_group_levels_retained": "",
                "n_race_grouped_levels_retained": np.nan,
                "race_grouped_levels_retained": "",
                "n_language_grouped_levels_retained": np.nan,
                "language_grouped_levels_retained": "",
                "n_matched_strata": np.nan,
                "min_pairs_per_stratum": np.nan,
                "median_pairs_per_stratum": np.nan,
                "max_pairs_per_stratum": np.nan,
                "min_total_per_stratum": np.nan,
                "median_total_per_stratum": np.nan,
                "max_total_per_stratum": np.nan,
                "demo_cols_retained": "",
                "demo_cols_dropped": "",
                "n_demo_cols_retained": np.nan,
                "n_demo_cols_dropped": np.nan,
                "OR_GEP": np.nan,
                "CI_low": np.nan,
                "CI_high": np.nan,
                "p_value": np.nan,
                "Converged": False,
                "RegularizedFallback": np.nan,
                "Error": f"{type(e).__name__}: {e}"
            })

    results = pd.DataFrame(run_rows)

    if strata_tables:
        strata_all = pd.concat(strata_tables, ignore_index=True)
    else:
        strata_all = pd.DataFrame()

    return results, strata_all


def summarize_repeated_results_with_diagnostics(results_df, label):
    valid = results_df.dropna(subset=["OR_GEP"]).copy()

    if valid.empty:
        print(results_df[["seed", "Error"]].head(20).to_string(index=False))
        raise ValueError("No valid runs were produced.")

    summary = pd.DataFrame([{
        "Analysis": label,
        "Runs": len(results_df),
        "ValidRuns": len(valid),

        "Mean_N": valid["n_total"].mean(),
        "SD_N": valid["n_total"].std(),
        "Min_N": valid["n_total"].min(),
        "Median_N": valid["n_total"].median(),
        "Max_N": valid["n_total"].max(),

        "Mean_GEP_N": valid["n_gep"].mean(),
        "Mean_NGEP_N": valid["n_ngep"].mean(),

        "Mean_matched_strata": valid["n_matched_strata"].mean(),
        "SD_matched_strata": valid["n_matched_strata"].std(),
        "Min_matched_strata": valid["n_matched_strata"].min(),
        "Median_matched_strata": valid["n_matched_strata"].median(),
        "Max_matched_strata": valid["n_matched_strata"].max(),

        "Mean_age_levels_retained": valid["n_age_group_levels_retained"].mean(),
        "Mean_race_levels_retained": valid["n_race_grouped_levels_retained"].mean(),
        "Mean_language_levels_retained": valid["n_language_grouped_levels_retained"].mean(),

        "Median_pairs_per_stratum": valid["median_pairs_per_stratum"].median(),
        "Min_pairs_per_stratum_over_runs": valid["min_pairs_per_stratum"].min(),
        "Max_pairs_per_stratum_over_runs": valid["max_pairs_per_stratum"].max(),

        "Mean_OR_GEP": valid["OR_GEP"].mean(),
        "SD_OR_GEP": valid["OR_GEP"].std(),
        "Median_OR_GEP": valid["OR_GEP"].median(),
        "P2.5_OR_GEP": valid["OR_GEP"].quantile(0.025),
        "P97.5_OR_GEP": valid["OR_GEP"].quantile(0.975),

        "Mean_p_value": valid["p_value"].mean(),
        "Prop_p_lt_0.05": (valid["p_value"] < 0.05).mean(),

        "Converged_runs": int(valid["Converged"].sum()),
        "RegularizedFallback_runs": int(valid["RegularizedFallback"].sum())
    }])

    return summary


# ------------------------------------------------
# RUN DIAGNOSTIC BALANCED ANALYSES
# ------------------------------------------------
strata_cols = ["age_group", "race_grouped", "language_grouped"]

rep_inc_diag, strata_inc_diag = repeat_balanced_analysis_with_diagnostics(
    GEP_df,
    outcome_col="label",
    strata_cols=strata_cols,
    analysis_label="Including misgendering",
    n_repeats=100,
    seed_start=1000
)

rep_exc_diag, strata_exc_diag = repeat_balanced_analysis_with_diagnostics(
    GEP_df,
    outcome_col="label_exclude_misgendering",
    strata_cols=strata_cols,
    analysis_label="Excluding misgendering",
    n_repeats=100,
    seed_start=2000
)

summary_inc_diag = summarize_repeated_results_with_diagnostics(
    rep_inc_diag,
    "Balanced repeated analysis (including misgendering)"
)

summary_exc_diag = summarize_repeated_results_with_diagnostics(
    rep_exc_diag,
    "Balanced repeated analysis (excluding misgendering)"
)

summary_both_diag = pd.concat([summary_inc_diag, summary_exc_diag], ignore_index=True)
rep_both_diag = pd.concat([rep_inc_diag, rep_exc_diag], ignore_index=True)
strata_both_diag = pd.concat([strata_inc_diag, strata_exc_diag], ignore_index=True)

print("\n=== Balanced repeated summary with diagnostics ===")
print(summary_both_diag.round(4).T)

print("\n=== First few diagnostic runs ===")
print(rep_both_diag.head().T)

print("\n=== First few retained strata ===")
print(strata_both_diag[strata_both_diag["retained"]].head(20))

# ------------------------------------------------
# SAVE FILES
# ------------------------------------------------
rep_inc_diag.to_csv(os.path.join(OUT_DIR, "balanced_runs_including_misgendering_with_diagnostics.csv"), index=False)
rep_exc_diag.to_csv(os.path.join(OUT_DIR, "balanced_runs_excluding_misgendering_with_diagnostics.csv"), index=False)
rep_both_diag.to_csv(os.path.join(OUT_DIR, "balanced_runs_both_outcomes_with_diagnostics.csv"), index=False)

strata_inc_diag.to_csv(os.path.join(OUT_DIR, "balanced_strata_including_misgendering.csv"), index=False)
strata_exc_diag.to_csv(os.path.join(OUT_DIR, "balanced_strata_excluding_misgendering.csv"), index=False)
strata_both_diag.to_csv(os.path.join(OUT_DIR, "balanced_strata_both_outcomes.csv"), index=False)

summary_both_diag.to_csv(os.path.join(OUT_DIR, "balanced_repeated_summary_with_diagnostics.csv"), index=False)

print("\nSaved diagnostic outputs to:")
print(OUT_DIR)